<a href="https://colab.research.google.com/github/run-llama/llama_index/blob/main/docs/examples/usecases/codebase_learning_assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Codebase Learning Assistant

This notebook shows how to build a small codebase learning assistant with LlamaIndex. The goal is not only to answer questions over code chunks, but also to help a new contributor form a staged learning path through an unfamiliar repository.

The pattern is useful when you want to:

- ingest source files from a repository or local project,
- preserve metadata such as file path, extension, and module,
- retrieve relevant implementation details, and
- synthesize a contributor-friendly learning roadmap.

If you are opening this notebook in Colab, install LlamaIndex first.

In [ ]:
%pip install llama-index llama-index-llms-openai

## Create a tiny sample repository

To keep the notebook self-contained, we create a toy repository. In a real workflow, replace `sample_repo` with a cloned GitHub repository or a local project directory.

In [ ]:
from pathlib import Path

repo_dir = Path("sample_repo").resolve()
(repo_dir / "app").mkdir(parents=True, exist_ok=True)
(repo_dir / "tests").mkdir(parents=True, exist_ok=True)

(repo_dir / "README.md").write_text(
    """# TaskFlow\n\nTaskFlow is a small FastAPI service that accepts tasks, stores them in memory, and exposes a worker loop for processing pending work.\n""",
    encoding="utf-8",
)
(repo_dir / "app" / "models.py").write_text(
    """from pydantic import BaseModel\n\n\nclass Task(BaseModel):\n    id: str\n    title: str\n    status: str = \"pending\"\n""",
    encoding="utf-8",
)
(repo_dir / "app" / "store.py").write_text(
    """from .models import Task\n\n\nclass TaskStore:\n    def __init__(self):\n        self._tasks: dict[str, Task] = {}\n\n    def add(self, task: Task) -> Task:\n        self._tasks[task.id] = task\n        return task\n\n    def pending(self) -> list[Task]:\n        return [task for task in self._tasks.values() if task.status == \"pending\"]\n""",
    encoding="utf-8",
)
(repo_dir / "app" / "main.py").write_text(
    """from fastapi import FastAPI\n\nfrom .models import Task\nfrom .store import TaskStore\n\napp = FastAPI()\nstore = TaskStore()\n\n\n@app.post(\"/tasks\")\ndef create_task(task: Task) -> Task:\n    return store.add(task)\n\n\n@app.get(\"/tasks/pending\")\ndef list_pending() -> list[Task]:\n    return store.pending()\n""",
    encoding="utf-8",
)
(repo_dir / "tests" / "test_store.py").write_text(
    """from app.models import Task\nfrom app.store import TaskStore\n\n\ndef test_pending_tasks():\n    store = TaskStore()\n    store.add(Task(id=\"1\", title=\"write docs\"))\n    assert len(store.pending()) == 1\n""",
    encoding="utf-8",
)

## Load code files with metadata

`SimpleDirectoryReader` can load local files and attach metadata. For code repositories, path-like metadata is important because contributors often ask questions about modules, entry points, and tests.

In [ ]:
from llama_index.core import SimpleDirectoryReader


def file_metadata(file_path: str) -> dict:
    path = Path(file_path).resolve()
    relative_path = path.relative_to(repo_dir)
    return {
        "file_path": str(relative_path),
        "extension": path.suffix,
        "module": relative_path.parts[0] if len(relative_path.parts) > 1 else "root",
    }


documents = SimpleDirectoryReader(
    input_dir=str(repo_dir),
    recursive=True,
    required_exts=[".py", ".md"],
    file_metadata=file_metadata,
).load_data()

for document in documents:
    print(document.metadata)

## Build an index

Use a splitter that keeps source files small enough for retrieval while preserving metadata on every node.

In [ ]:
from llama_index.core import Settings, VectorStoreIndex
from llama_index.core.node_parser import SentenceSplitter
from llama_index.llms.openai import OpenAI

Settings.llm = OpenAI(model="gpt-4o-mini")
Settings.node_parser = SentenceSplitter(chunk_size=512, chunk_overlap=64)

index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine(similarity_top_k=6)

## Ask contributor-oriented questions

The first set of questions should map the project before asking for implementation details.

In [ ]:
print(
    query_engine.query(
        "What is the main purpose of this repository? Mention the key files a new contributor should inspect first."
    )
)

In [ ]:
print(
    query_engine.query(
        "Trace the main execution flow from the API layer to the storage layer. Include file paths."
    )
)

## Generate a staged learning path

A learning assistant can turn retrieved context into a roadmap. This makes the output useful for onboarding, not just question answering.

In [ ]:
learning_path_prompt = """
Create a four-stage learning path for a new contributor to this repository.

Use this structure:
1. First map the project: purpose, folders, and entry points.
2. Run the main flow: how data moves through the system.
3. Study the core implementation patterns: abstractions and tradeoffs.
4. Apply one lesson: a small exercise the contributor can implement.

For each stage, include the files to inspect and one concrete question to answer.
"""

print(query_engine.query(learning_path_prompt))

## Production notes

For a larger repository, this baseline can be extended with:

- a code parser that extracts symbols, imports, and call relationships,
- a property graph index for module and symbol relationships,
- metadata filters for language, folder, or test files,
- reranking for high-signal code snippets, and
- separate prompts for architecture overview, execution flow, implementation patterns, and reusable takeaways.